In [1]:
# Cell 1: environment check
import sys, torch, ultralytics
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

print(f"python      {sys.executable}")
print(f"ultralytics {ultralytics.__version__}")
print(f"torch       {torch.__version__}")
print(f"device      {DEVICE}")

python      /Users/nikhilkunapareddy/.venvs/loonet-train/bin/python
ultralytics 8.4.138
torch       2.14.0
device      mps


In [2]:
# Cell 2: write train/val lists — 80/20 within each species, so val has all four
from pathlib import Path
import random, yaml

DS = next(p / "data/annotated/vegnet_v1"
        for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_YAML = DS / "data.yaml"      # written by scripts/autolabel_vegnet.py, names included

imgs = sorted(p for p in (DS / "images").glob("*.jpg")
            if (DS / "labels" / f"{p.stem}.txt").exists())
by_species = {}
for p in imgs:
    by_species.setdefault(p.stem.rsplit("_", 1)[0], []).append(p)

train, val = [], []
for species, ps in sorted(by_species.items()):
    random.Random(0).shuffle(ps)
    k = round(len(ps) * 0.2)
    val += ps[:k]; train += ps[k:]
    print(f"{species:15s} train {len(ps) - k:3d}   val {k:3d}")

(DS / "val.txt").write_text("\n".join(str(p) for p in val))
(DS / "train.txt").write_text("\n".join(str(p) for p in train))
print(f"train {len(train)}   val {len(val)}   (of {len(imgs)} labeled)")
print(yaml.safe_load(DATA_YAML.read_text())["names"])


loosestrife     train  26   val   7
phragmites      train  26   val   7
water_chestnut  train  26   val   7
yellow_iris     train  26   val   7
train 104   val 28   (of 132 labeled)
{0: 'purple loosestrife', 1: 'common reed', 2: 'water chestnut', 3: 'yellow flag iris'}


In [3]:
# Cell 3: fine-tune YOLO26s — one line per epoch, val detections snapshotted each epoch
import logging, time, cv2, numpy as np
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils import LOGGER

LOGGER.setLevel(logging.WARNING)      # silences the per-batch progress bars
ROOT = DS.parents[2]

val_imgs = [Path(p) for p in (DS / "val.txt").read_text().split()]
WATCH = [next(p for p in val_imgs if p.stem.startswith(s)) for s in sorted(by_species)]
last_t = [time.time()]

def on_epoch(trainer):
    if trainer.epoch >= trainer.epochs:       # skip the duplicate call from final eval
        return
    m = trainer.metrics
    losses = " ".join(f"{k.split('/')[-1].split('_')[0]} {v:.3f}"
                    for k, v in trainer.label_loss_items(trainer.tloss).items())
    print(f"epoch {trainer.epoch + 1:>3}/{trainer.epochs} | {losses} | "
        f"P {m['metrics/precision(B)']:.3f} R {m['metrics/recall(B)']:.3f} "
        f"mAP50 {m['metrics/mAP50(B)']:.3f} mAP50-95 {m['metrics/mAP50-95(B)']:.3f} | "
        f"{time.time() - last_t[0]:.0f}s", flush=True)
    last_t[0] = time.time()

    out = Path(trainer.save_dir) / "progress"; out.mkdir(exist_ok=True)
    snap = YOLO(trainer.last, verbose=False)
    tiles = [cv2.resize(snap.predict(i, imgsz=640, conf=0.25, device=DEVICE,
                                    verbose=False)[0].plot(), (480, 360)) for i in WATCH]
    grid = np.vstack([np.hstack(tiles[:2]), np.hstack(tiles[2:])])
    cv2.putText(grid, f"epoch {trainer.epoch + 1}", (12, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.imwrite(str(out / f"epoch_{trainer.epoch + 1:03d}.jpg"), grid)

model = YOLO("yolo26s.pt")            # end-to-end head: no NMS at inference
model.add_callback("on_fit_epoch_end", on_epoch)

results = model.train(
    data=str(DATA_YAML), epochs=100, patience=30, imgsz=640, batch=16,
    cache=True, device=DEVICE, seed=0,
    project=str(ROOT / "runs"), name="vegnet_v1_yolo26s",
)

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
train: Caching images (0.1GB RAM): 100% ━━━━━━━━━━━━ 104/104 1.4Kit/s 0.1s
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.0GB RAM): 100% ━━━━━━━━━━━━ 28/28 1.5Kit/s 0.0s
epoch   1/100 | box 1.144 cls 21.889 l1 0.026 | P 0.351 R 0.219 mAP50 0.135 mAP50-95 0.093 | 1595s
epoch   2/100 | box 1.039 cls 12.682 l1 0.022 | P 0.311 R 0.154 mAP50 0.124 mAP50-95 0.088 | 200s
epoch   3/100 | box 1.039 cls 9.615 l1 0.023 | P 0.368 R 0.098 mAP50 0.101 mAP50-95 0.062 | 149s
epoch   4/100 | box 1.154 cls 7.050 l1 0.026 | P 0.182 R 0.383 mAP50 0.178 mAP50-95 0.058 | 109s
epoch   5/100 | box 1.139 cls 4.987 l1 0.025 | P 0.097 R 0.209 mAP50 0.115 mAP50-95 0.057 | 29s
epoch   6/100 | box 1.181 cls 4.204 l1 0.024 | P 0.168 R 0.157 mA

/Users/nikhilkunapareddy/.venvs/loonet-train/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/nikhilkunapareddy/.venvs/loonet-train/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation

In [4]:
# Cell 4: full val plots (every val image, not just the first 16)
best = YOLO(ROOT / "runs/vegnet_v1_yolo26s/weights/best.pt")
m = best.val(data=str(DATA_YAML), batch=16, imgsz=640, device=DEVICE,
            plots=True, project=str(ROOT / "runs"), name="vegnet_v1_yolo26s_val")
print(f"P {m.box.mp:.3f}  R {m.box.mr:.3f}  mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}")

P 0.816  R 0.744  mAP50 0.773  mAP50-95 0.572


In [5]:
# Cell 5: save v1 — runs/*/weights is gitignored, models/ is versioned
import shutil
V1 = ROOT / "models/vegnet/v1"; V1.mkdir(parents=True, exist_ok=True)
shutil.copy2(ROOT / "runs/vegnet_v1_yolo26s/weights/best.pt", V1 / "best.pt")
shutil.copy2(DATA_YAML, V1 / "data.yaml")
print(f"{V1 / 'best.pt'}  {(V1 / 'best.pt').stat().st_size / 1e6:.1f} MB")


/Users/nikhilkunapareddy/Documents/loonet/models/vegnet/v1/best.pt  20.3 MB
